# Research Assistant with Tool Calling (Agentic Workflow)

Simulates an agentic AI workflow using OpenAI-style tool calling:
- Planner LLM decides which tools to call and in what order
- Tools: search_topics, summarize_text, get_citation, format_report
- Loop: chat with tools → handle tool_calls → append results → repeat until done

In [13]:
import os
import sys
import json
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv

_cwd = os.getcwd()
_candidates = [_cwd, os.path.join(_cwd, "week8", "community_contributions", "najeeb"), os.path.join(_cwd, "community_contributions", "najeeb")]
for _p in _candidates:
    if os.path.isfile(os.path.join(_p, "knowledge_base.py")):
        if _p not in sys.path:
            sys.path.insert(0, _p)
        os.chdir(_p)
        break

from knowledge_base import KNOWLEDGE_BASE

load_dotenv(override=True)


API_KEY = os.getenv("OPENAI_API_KEY")
BASE_URL = "https://openrouter.ai/api/v1"
MODEL ="gpt-4o-mini"
if API_KEY and API_KEY.startswith("sk-or-") and not BASE_URL:
    BASE_URL = "https://openrouter.ai/api/v1"

openai = OpenAI(base_url=BASE_URL, api_key=API_KEY) if BASE_URL else OpenAI(api_key=API_KEY)
print(f"API key set: {bool(API_KEY)}, Model: {MODEL}")

API key set: True, Model: gpt-4o-mini


## Tool implementations

In [14]:
def search_topics(query: str) -> str:
    """Search for articles/snippets by topic keywords. Returns JSON string of results."""
    query_lower = query.lower()
    results = []
    for item in KNOWLEDGE_BASE:
        if any(word in item["topic"] or word in item["text"].lower() for word in query_lower.split()):
            results.append({"id": item["id"], "topic": item["topic"], "snippet": item["text"][:200] + "..."})
    return json.dumps(results if results else [{"id": "none", "topic": "none", "snippet": "No matches."}])


def summarize_text(text: str) -> str:
    """Summarize a long text in 2-3 sentences."""
    try:
        r = openai.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": f"Summarize in 2-3 sentences:\n\n{text}"}],
            max_tokens=150,
        )
        return r.choices[0].message.content or text[:200]
    except Exception as e:
        return f"[Summary placeholder]: {text[:200]}... (Error: {e})"


def get_citation(source_id: str) -> str:
    """Get a citation for a source by id."""
    for item in KNOWLEDGE_BASE:
        if item["id"] == source_id:
            return f"{item['author']} ({item['year']}). {item['topic']}. [ID: {source_id}]"
    return f"Unknown source (ID: {source_id})"


def format_report(title: str, bullets: list) -> str:
    """Format the final report with title and bullets."""
    lines = [f"# {title}", ""] + [f"- {b}" for b in bullets]
    return "\n".join(lines)

## Tool definitions (OpenAI function-calling format)

In [15]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_topics",
            "description": "Search for articles or snippets by topic keywords. Returns a list of matching items with id, topic, and snippet.",
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string", "description": "Search query"}},
                "required": ["query"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "summarize_text",
            "description": "Summarize a long text in 2-3 sentences.",
            "parameters": {
                "type": "object",
                "properties": {"text": {"type": "string", "description": "The text to summarize"}},
                "required": ["text"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_citation",
            "description": "Get a citation string for a source by its id (e.g. src1, src2).",
            "parameters": {
                "type": "object",
                "properties": {"source_id": {"type": "string", "description": "The source id from search results"}},
                "required": ["source_id"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "format_report",
            "description": "Format the final report with a title and a list of bullet points.",
            "parameters": {
                "type": "object",
                "properties": {
                    "title": {"type": "string", "description": "Report title"},
                    "bullets": {"type": "array", "items": {"type": "string"}, "description": "List of bullet point strings"},
                },
                "required": ["title", "bullets"],
                "additionalProperties": False,
            },
        },
    },
]

TOOL_IMPLS = {
    "search_topics": search_topics,
    "summarize_text": summarize_text,
    "get_citation": get_citation,
    "format_report": format_report,
}


## Agentic loop

In [16]:
def handle_tool_calls(message):
    results = []
    for tc in message.tool_calls:
        name = tc.function.name
        args = json.loads(tc.function.arguments or "{}")
        fn = TOOL_IMPLS.get(name)
        out = fn(**args) if fn else json.dumps({"error": f"Unknown tool: {name}"})
        results.append({"role": "tool", "content": out if isinstance(out, str) else json.dumps(out), "tool_call_id": tc.id})
    return results


def run_research_assistant(user_query: str, history: list | None = None) -> str:
    """Run the research assistant. history is a list of [user_msg, assistant_msg] from prior turns."""
    system = "You are a research assistant. Use your tools to: 1) search for relevant content by topic, 2) summarize any long text, 3) get citations for sources by id, 4) format a short report with title and bullets. End by calling format_report, then reply briefly to the user."
    messages = [{"role": "system", "content": system}]
    if history:
        for user_msg, assistant_msg in history:
            if user_msg:
                messages.append({"role": "user", "content": user_msg})
            if assistant_msg:
                messages.append({"role": "assistant", "content": assistant_msg})
    messages.append({"role": "user", "content": user_query})
    while True:
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS)
        choice = response.choices[0]
        msg = choice.message
        if choice.finish_reason == "tool_calls":
            messages.append({"role": "assistant", "content": msg.content or "", "tool_calls": [{"id": t.id, "type": "function", "function": {"name": t.function.name, "arguments": t.function.arguments}} for t in msg.tool_calls]})
            for tr in handle_tool_calls(msg):
                messages.append(tr)
            continue
        return (msg.content or "").strip()

## Run

In [ ]:
# Use the Gradio Chat Interface below to interact with the research assistant.

## Gradio Chat Interface

In [17]:

def chat(message, history):
    """Gradio chat fn: takes user message and conversation history, returns assistant reply."""
    if not message or not message.strip():
        return ""
    return run_research_assistant(message.strip(), history=history)


demo = gr.ChatInterface(
    chat,
    title="Research Assistant",
    description="Ask for research on topics (e.g. attention mechanisms, transformers). I'll search, summarize, and cite sources.",
    examples=[
        "Research 'attention mechanisms in transformers' and give me a short report with citations.",
        "Summarize what you know about multi-head attention and positional encoding.",
    ],
)

demo.launch()

c:\Users\Najeeb\Projects\llm_engineering\.venv\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
